# Financial classification — synthetic corpus
All 12,600 records are invented, using the 70 existing active categories.
Docker and SQL are not required. Earlier private datasets are not loaded.
Restart the kernel if an earlier version of this notebook was open.
First install dependencies in the notebook kernel: `%pip install -e "..[test]"`
from the notebooks directory, or `%pip install -e ".[test]"` from the project root;
then restart the kernel.
Synthetic metrics do not demonstrate quality on real transactions.
Save the notebook without outputs before committing.
Descriptions and category labels remain in Portuguese, the classification task's source language.


In [ ]:
from pathlib import Path
import os, sys, gc
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if root is None: raise RuntimeError("Open the notebook inside the project directory")
os.chdir(root)
sys.path.insert(0, str(root / "src"))
from denarius_model import ROOT
from denarius_model.config import load_config, read_json, dataset_path, assert_dataset_version
from denarius_model.diagnostics import diagnose
config = load_config()
# Set False and restart the kernel for full-data training.
QUICK_RUN = True
experiment = "synthetic-quick-001" if QUICK_RUN else "synthetic-001"
if QUICK_RUN:
    config["training"].update(epochs=1, gradient_accumulation=4,
                              train_per_category=2, validation_per_category=1)
RUN_BASELINE_HF = not QUICK_RUN
RUN_OLLAMA_BASELINE = False
RUN_FULL_VALIDATION = False
RUN_FINAL = False
RUN_EXPORT = False
EVAL_PER_CATEGORY = 1 if QUICK_RUN else (None if RUN_FULL_VALIDATION else config["dataset"]["development_eval_per_category"])
if QUICK_RUN and (RUN_FINAL or RUN_FULL_VALIDATION):
    raise ValueError("Disable QUICK_RUN before full validation or final testing")
print("Mode:", "quick demonstration" if QUICK_RUN else "full training")
print("Experiment:", experiment)
diagnose()

## Deterministic preparation and execution mode
`QUICK_RUN=True` is the default: 140 training examples (two per category), one epoch,
70 validation examples for training loss, and 70 examples for generation-based evaluation.
The untuned HF baseline is skipped to save time. The public dataset files remain intact.
This demonstrates training, predictions, metrics and reporting; quality is not representative of full training.

For full training, set `QUICK_RUN=False`, choose a fresh experiment ID if needed,
restart the kernel and run the notebook again. This uses all 8,400 training and
1,400 validation examples for three epochs, including the untuned HF baseline.
`RUN_FULL_VALIDATION=True` also evaluates all validation examples with generated responses.
Final tests remain disabled until the full candidate is frozen.

Generation uses only the versioned taxonomy and synthetic rules. Hard-test scenarios
reserve fictional counterparties and formats; label-defining brands remain known.


In [ ]:
from denarius_model.synthetic import audit_public_dataset
from denarius_model.dataset import prepare
# Regenerate only data/synthetic; earlier real datasets are neither read nor replaced.
stats = prepare()
categories = read_json(dataset_path("processed/categories.json"))
from denarius_model.train import training_rows
train_rows = training_rows(config, "train")
print("Selected training examples:", len(train_rows))
print("Dataset:", stats["dataset_version"])
print("Splits:", stats["sizes"])
assert stats["source"] == "synthetic"
assert all(r["synthetic"] for r in train_rows)

## Exploration and distribution
The corpus is balanced by construction. It does not reproduce real usage frequencies.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.Series(stats["categories"]).sort_values().plot.barh(figsize=(12, 18))
plt.tight_layout()
# Display only six synthetic examples, with no private transactions.
display(pd.DataFrame(train_rows)[["description", "amount", "direction", "category"]].head(6))

## Prompts and token budget
Only description, amount and direction are used as features. Synthetic identifiers are used only to validate splits.


In [ ]:
from denarius_model.prompting import prompt, example
from denarius_model.config import model_source
from transformers import AutoTokenizer
synthetic = {"description":"ALDI compras supermercado SYN-LAROMA", "amount":"37.25",
             "direction":"debito_contabilistico", "category":"Despesas Correntes / Compras no ALDI"}
print(example(synthetic, categories)["completion"])
tokenizer = AutoTokenizer.from_pretrained(model_source(config["model"], True), local_files_only=True)
lengths = [len(tokenizer.encode(example(r,categories)["prompt"], add_special_tokens=False)) +
           len(tokenizer.encode(example(r,categories)["completion"], add_special_tokens=False)) + 1
           for r in train_rows]
print("Maximum training sequence length:", max(lengths), "/", config["model"]["max_length"])
assert max(lengths) <= config["model"]["max_length"], "Increase context length or revise the format without truncating categories"
del tokenizer

## Validation baselines
TF-IDF is fitted on the training split only. In quick mode, the untuned HF baseline
is skipped and generation-based evaluation uses one validation example per category.
All evaluated models use the same validation sample. Full mode enables the HF baseline;
RUN_FULL_VALIDATION=True evaluates the complete validation split. Ollama is optional.


In [ ]:
from denarius_model.evaluate import evaluate, ml_baseline, report
from denarius_model.inference import HFClassifier, OllamaClassifier, classify
ml = ml_baseline()
ml_validation = evaluate(ml, "validation", "ml", experiment, per_category=EVAL_PER_CATEGORY)
if RUN_OLLAMA_BASELINE:
    ollama_base = OllamaClassifier(config["model"]["ollama_baseline"], native_template=True)
    try:
        ollama_validation = evaluate(ollama_base, "validation", "ollama-base", experiment, per_category=EVAL_PER_CATEGORY)
    finally:
        ollama_base.unload()
base_validation = None
if RUN_BASELINE_HF:
    base = HFClassifier(config["model"])
    base_validation = evaluate(base, "validation", "base", experiment, per_category=EVAL_PER_CATEGORY)
    del base

gc.collect()
import torch
torch.cuda.empty_cache()


## LoRA / QLoRA and training
Quick mode uses one epoch and gradient accumulation 4 on a balanced training subset.
Full mode uses the training YAML settings and complete training/validation splits.
Both use checkpointing, completion-only loss and early stopping on their selected
validation split. The manifest records actual subset sizes and hashes.
A matching existing adapter is reused. To retrain, choose a fresh experiment ID;
quick and full runs use separate IDs and never overwrite each other.


In [ ]:
from denarius_model.train import train
manifest_path = ROOT / "outputs" / experiment / "experiment.json"
adapter_dir = ROOT / "models" / experiment / "adapter"
if manifest_path.exists() and adapter_dir.exists():
    assert_dataset_version(experiment)
    previous = read_json(manifest_path)
    assert previous["configuration"] == config, "Configuration changed: use a new experiment ID"
    adapter_path = str(adapter_dir)
    print("Reusing existing adapter:", adapter_path)
else:
    adapter_path = train(config, experiment)
gc.collect(); torch.cuda.empty_cache()

## Training curves and validation evaluation


In [ ]:
manifest = read_json(manifest_path)
history = pd.DataFrame(manifest["metrics"])
for column in ["loss", "eval_loss"]:
    if column in history:
        history.dropna(subset=[column]).plot(x="epoch", y=column)
tuned = HFClassifier(config["model"], adapter_path)
tuned_validation = evaluate(tuned, "validation", "tuned", experiment, per_category=EVAL_PER_CATEGORY)
results = {"TF-IDF": ml_validation, "Fine-tuned": tuned_validation}
if base_validation is not None:
    results["HF Base"] = base_validation
assert len({result["evaluation_rows_hash"] for result in results.values()}) == 1
comparison = pd.DataFrame(results).T
comparison[["sample_size", "accuracy", "macro_f1", "weighted_f1", "invalid_outputs"]]


## Confusion matrix and per-category metrics
Categories come from the existing taxonomy; support depends on the exploratory sample or full split.


In [ ]:
display(pd.DataFrame(tuned_validation["per_category"]).T)
fig, ax = plt.subplots(figsize=(12, 12))
im = ax.imshow(tuned_validation["confusion_matrix"], cmap="Blues")
fig.colorbar(im, ax=ax)
ax.set_xlabel("Predicted category index"); ax.set_ylabel("True category index")
ax.set_title("Synthetic validation — indices listed in confusion_labels")
plt.tight_layout()
display(pd.Series(tuned_validation["confusion_labels"], name="confusion_labels"))

## Errors and manual inference
The examples below are synthetic only. They do not represent private transactions.


In [ ]:
predictions = read_json(ROOT / "reports" / experiment / "tuned_validation_predictions.json")
errors = [r for r in predictions if r["row"]["category"] != r["prediction"]]
display(pd.DataFrame([{ "description":r["row"]["description"], "true_category":r["row"]["category"],
                       "prediction":r["prediction"]} for r in errors[:6]]))
try: print(classify(tuned, synthetic, categories))
except ValueError: print("Invalid output")

## Sensitivity
Counterfactual amount/direction ablations may justify a different category; a change in prediction does not automatically indicate an error. scripts/robustness.py evaluates the complete validation split separately.


In [ ]:
variants = [synthetic, {**synthetic,"amount":"1200.00"},
            {**synthetic,"direction":"credito_contabilistico"},
            {**synthetic,"description":"aldi compras supermercado syn-laroma"}]
for item in variants:
    try: print(classify(tuned, item, categories))
    except ValueError: print("Invalid output")

## Final test and hard test
Freeze the candidate and hyperparameters before enabling RUN_FINAL. Do not use
these results for tuning. These are complete synthetic tests, not evaluation on real data.


In [ ]:
if RUN_FINAL and QUICK_RUN:
    raise ValueError("Final tests require a frozen full-training candidate; disable QUICK_RUN")
if RUN_FINAL:
    for split in ["test", "hard_test"]:
        evaluate(tuned, split, "tuned", experiment, final=True)
        evaluate(ml, split, "ml", experiment, final=True)
    del tuned; gc.collect(); torch.cuda.empty_cache()
    base = HFClassifier(config["model"])
    for split in ["test", "hard_test"]:
        evaluate(base, split, "base", experiment, final=True)
    del base; gc.collect(); torch.cuda.empty_cache()
report(experiment)
print("Local report:", ROOT / "reports" / experiment / "evaluation.md")

## Merging and export preparation
Merge on the CPU in FP32. RUN_EXPORT does not automatically convert to GGUF.
Run the llama.cpp and ollama create commands from the README, recording versions locally.
Compare Q4_K_M and F16 with the same backend and schema to isolate quantization effects.


In [ ]:
if RUN_EXPORT:
    if "tuned" in globals():
        del tuned; gc.collect(); torch.cuda.empty_cache()
    from denarius_model.export import merge
    print(merge(config, experiment))

## Ollama inference and benchmarking
After creating denarius-movement, uncomment the code below. raw=true reproduces the trained completion prompt; the schema constrains categories. Do not infer readiness for real data.


In [ ]:
# ollama = OllamaClassifier()
# print(classify(ollama, synthetic, categories))
# evaluate(ollama, "validation", "ollama", experiment, per_category=EVAL_PER_CATEGORY)
# ollama.unload()
# report(experiment)
# Cold/warm benchmark: python scripts/benchmark_ollama.py --model denarius-movement